# Week 1 · Math essentials for ML

This is the theory block (90 minutes a day), next to the Python work. It covers only the math the ML and DL weeks actually use. Each idea is explained in words, worked through by hand, then **checked with NumPy**. The drills at the end of each section are the kind of thing the exam asks you to do on paper. Pace: sections 1–2 Mon, 3–4 Tue, 5–6 Wed, 6–7 Thu, 7 Fri, 8–9 Sat.

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

## Words you'll see
- **$P(A)$**: The probability of event $A$, a number from 0 to 1.
- **$P(A \mid B)$**: The probability of $A$ **given that** $B$ happened. Read the bar as “given”.
- **random variable**: A number whose value depends on chance, e.g. the result of a die roll. Written with a capital letter, like $X$.
- **$\mathbb{E}[X]$**: Expected value: the long-run average of $X$. Also written $\mu$.
- **$\operatorname{Var}(X)$**: Variance: how spread out $X$ is. Its square root is the standard deviation $\sigma$.
- **likelihood**: $L(\theta)$: how probable the **observed data** is, for a given choice of model parameters $\theta$.
- **gradient $\nabla L$**: The vector of partial derivatives $\left(\frac{\partial L}{\partial w_1}, \dots, \frac{\partial L}{\partial w_d}\right)$. It points in the direction where $L$ **increases** fastest.
- **loss**: A number $L$ that measures how wrong a model is. Training means making it small.

## 1. Probability and Bayes' rule

**Conditional probability.** Among the cases where $B$ happens, what fraction also have $A$?

$$
P(A \mid B) = \frac{P(A \cap B)}{P(B)}
$$

**Bayes' rule** flips the condition around. It's useful because $P(B \mid A)$ is often easy to know, while $P(A \mid B)$ is the thing you actually want:

$$
P(A \mid B) = \frac{P(B \mid A)\, P(A)}{P(B)}
$$

**Worked example.** 1% of people have a disease. The test catches 99% of sick people, so $P(+ \mid \text{sick}) = 0.99$, and wrongly says $+$ for 5% of healthy people, so $P(+ \mid \text{healthy}) = 0.05$. You test $+$. What is the chance you're sick?

| Step | Calculation | Value |
|---|---|---|
| $P(+ \cap \text{sick})$ | $0.99 \times 0.01$ | 0.0099 |
| $P(+ \cap \text{healthy})$ | $0.05 \times 0.99$ | 0.0495 |
| $P(+)$, the sum of the two | $0.0099 + 0.0495$ | 0.0594 |
| $P(\text{sick} \mid +)$ | $0.0099 / 0.0594$ | **$\approx 0.167$** |

Only about 17%, because healthy people are so much more common that their false positives outnumber the true positives. Check it by simulating a million people:

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
n = 1_000_000
sick = rng.random(n) < 0.01                              # True for about 1% of people
positive = np.where(sick, rng.random(n) < 0.99,           # sick: + with prob 0.99
                          rng.random(n) < 0.05)           # healthy: + with prob 0.05
print(round(sick[positive].mean(), 3))                    # → close to 0.167

This is exactly how a **naive Bayes classifier** works (Week 2): $P(\text{class} \mid \text{features}) \propto P(\text{features} \mid \text{class})\, P(\text{class})$.

**Drill (on paper).** 20% of emails are spam. The word “free” appears in 60% of spam and in 5% of normal emails. An email contains “free”. What is $P(\text{spam} \mid \text{free})$?

<details><summary>Worked answer</summary>

$P(\text{free} \cap \text{spam}) = 0.6 \times 0.2 = 0.12$. $P(\text{free} \cap \text{normal}) = 0.05 \times 0.8 = 0.04$. $P(\text{free}) = 0.16$. So $P(\text{spam} \mid \text{free}) = 0.12 / 0.16 = \mathbf{0.75}$.

</details>

## 2. Expectation and variance

The **expected value** is the probability-weighted average. The **variance** is the average squared distance from the mean:

$$
\mathbb{E}[X] = \sum_x x\, P(x) \qquad \operatorname{Var}(X) = \mathbb{E}\big[(X - \mathbb{E}[X])^2\big]
$$

An easier formula to compute with:

$$
\operatorname{Var}(X) = \mathbb{E}[X^2] - \big(\mathbb{E}[X]\big)^2
$$

**Fair die:** $\mathbb{E}[X] = \frac{1+2+3+4+5+6}{6} = 3.5$. $\mathbb{E}[X^2] = \frac{1+4+9+16+25+36}{6} = \frac{91}{6} \approx 15.17$. So $\operatorname{Var}(X) \approx 15.17 - 3.5^2 = 15.17 - 12.25 \approx \mathbf{2.92}$.

In [ ]:
faces = np.arange(1, 7)
p = np.ones(6) / 6
EX = (faces * p).sum()
EX2 = (faces**2 * p).sum()
print(EX, round(EX2 - EX**2, 3))       # → 3.5 2.917

**Two rules you'll use all the time.** Adding a constant $b$ doesn't change the spread, but scaling by $a$ multiplies the variance by $a^2$:

$$
\mathbb{E}[aX + b] = a\,\mathbb{E}[X] + b \qquad \operatorname{Var}(aX + b) = a^2 \operatorname{Var}(X)
$$

**Drill (on paper).** $P(X=0) = 0.5$, $P(X=1) = 0.3$, $P(X=2) = 0.2$. Find $\mathbb{E}[X]$ and $\operatorname{Var}(X)$. Then find $\operatorname{Var}(3X + 1)$.

<details><summary>Worked answer</summary>

$\mathbb{E}[X] = 0 + 0.3 + 0.4 = 0.7$. $\mathbb{E}[X^2] = 0 + 0.3 + 0.8 = 1.1$. $\operatorname{Var}(X) = 1.1 - 0.49 = \mathbf{0.61}$. $\operatorname{Var}(3X + 1) = 9 \times 0.61 = \mathbf{5.49}$.

</details>

## 3. The Gaussian (normal) distribution

The bell curve $\mathcal{N}(\mu, \sigma^2)$, with mean $\mu$ (the center) and standard deviation $\sigma$ (the width):

$$
p(x) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\left(-\frac{(x - \mu)^2}{2\sigma^2}\right)
$$

About **68%** of values fall within $1\sigma$ of the mean, **95%** within $2\sigma$, and **99.7%** within $3\sigma$.

**Standardizing,** $z = \frac{x - \mu}{\sigma}$, turns any normal into $\mathcal{N}(0, 1)$. It's the same formula as the NumPy feature standardization.

In [ ]:
x = rng.normal(loc=170, scale=10, size=100_000)     # heights: μ = 170, σ = 10
inside_1sd = np.abs(x - 170) < 10
print(round(inside_1sd.mean(), 2))                   # → close to 0.68

**More than one variable:** a multivariate Gaussian $\mathcal{N}(\boldsymbol{\mu}, \Sigma)$ has a **mean vector** $\boldsymbol{\mu}$ and a **covariance matrix** $\Sigma$. The diagonal entry $\Sigma_{ii}$ is the variance of variable $i$, and the off-diagonal $\Sigma_{ij}$ is the covariance of $i$ and $j$ (positive means they tend to go up together).

In [ ]:
h = rng.normal(170, 10, 5000)
w = 0.9 * h + rng.normal(0, 5, 5000) - 90            # weight goes up with height
print(np.cov(h, w).round(0))                         # 2×2 covariance matrix
# → [[ 98.  88.]
#    [ 88. 104.]]   diagonal = the variances (in theory 100 and 106);
#                   off-diagonal = the covariance (in theory 90). Positive: they rise together

## 4. Maximum likelihood (MLE)

**The idea:** choose the parameters $\theta$ that make the data you actually observed **as probable as possible**: $\hat\theta = \arg\max_\theta L(\theta)$.

**Coin example.** You flip a coin 10 times and get 7 heads. If the probability of heads is $p$, the likelihood of exactly this data is:

$$
L(p) = p^7 (1 - p)^3
$$

**Step 1, take the log.** The log turns products into sums and has its maximum in the same place:

$$
\log L(p) = 7 \log p + 3 \log(1 - p)
$$

**Step 2, set the derivative to 0:**

$$
\frac{d}{dp}\log L = \frac{7}{p} - \frac{3}{1 - p} = 0 \;\Longrightarrow\; 7(1 - p) = 3p \;\Longrightarrow\; \hat p = \frac{7}{10} = 0.7
$$

That's just the fraction of heads, which is what intuition says too.

In [ ]:
p = np.linspace(0.01, 0.99, 99)
log_L = 7 * np.log(p) + 3 * np.log(1 - p)
print(round(p[np.argmax(log_L)], 2))     # → 0.7

**For a Gaussian**, the MLE of the mean is the **sample average**, $\hat\mu = \frac{1}{n}\sum_i x_i$, and the MLE of the variance is $\hat\sigma^2 = \frac{1}{n}\sum_i (x_i - \hat\mu)^2$.

**Why this matters for the whole course: every common loss function is an MLE in disguise.**

| Loss you minimize | Is MLE for |
|---|---|
| mean squared error $\frac{1}{n}\sum_i (y_i - \hat y_i)^2$ (regression) | a Gaussian noise model: $y = \hat y + \varepsilon$, $\varepsilon \sim \mathcal{N}(0, \sigma^2)$ |
| binary cross-entropy $-[y \log \hat p + (1-y)\log(1-\hat p)]$ (logistic regression) | a Bernoulli model of the label |
| cross-entropy $-\log \hat p_{\text{true}}$ (softmax classifier) | a categorical model of the label |

**Drill (on paper).** You see 3 heads in 12 flips. What is the MLE of $p$? Show the derivative step.

<details><summary>Worked answer</summary>

$\log L = 3 \log p + 9 \log(1 - p)$. The derivative is $\frac{3}{p} - \frac{9}{1 - p} = 0 \Rightarrow 3 - 3p = 9p \Rightarrow \hat p = \mathbf{0.25}$.

</details>

## 5. MAP: adding a prior belief

MLE uses only the data. **MAP** (maximum a posteriori) also uses a **prior** $P(\theta)$, which is what you believed before seeing the data. By Bayes' rule, MAP picks the peak of the posterior:

$$
P(\theta \mid \text{data}) \propto P(\text{data} \mid \theta)\, P(\theta) \qquad \hat\theta_{\text{MAP}} = \arg\max_\theta P(\theta \mid \text{data})
$$

For a coin, the standard prior is a **Beta**$(a, b)$ distribution. With $h$ heads in $n$ flips, the MAP estimate has a simple formula:

$$
\hat p_{\text{MAP}} = \frac{h + a - 1}{n + a + b - 2}
$$

With a $\text{Beta}(2, 2)$ prior (a mild belief that the coin is fair) and 7 heads in 10: $\hat p = \frac{7 + 1}{10 + 2} = \frac{8}{12} \approx \mathbf{0.667}$. That's pulled from 0.7 toward 0.5. With 700 heads in 1000, MAP gives $\frac{701}{1002} \approx 0.6996$, almost the same as MLE. **The more data you have, the less the prior matters.**

**The link to ML:** putting a Gaussian prior on a model's weights and doing MAP is the same as adding the penalty $\lambda \lVert w \rVert^2$ to the loss. That's **ridge regression** and **weight decay** (Weeks 2 and 6).

**Drill (on paper).** Same data as the last drill (3 heads in 12), with a $\text{Beta}(3, 3)$ prior. What is the MAP estimate, and which way did it move compared with the MLE?

<details><summary>Worked answer</summary>

$\hat p = \frac{3 + 2}{12 + 4} = \frac{5}{16} = \mathbf{0.3125}$. It moved from 0.25 toward 0.5, the center of the prior.

</details>

## 6. Linear algebra for ML

**Dot product:** $u \cdot v = \sum_i u_i v_i = u^\top v$. It's large when the two vectors point the same way, 0 when they're perpendicular, and negative when they point opposite ways. It's the basic operation of every model: a prediction is $\hat y = w^\top x + b$.

**Matrix × vector:** for $A \in \mathbb{R}^{m \times n}$ and $x \in \mathbb{R}^{n}$, the product $Ax \in \mathbb{R}^{m}$, and each entry $(Ax)_i$ is **row $i$ of $A$** dotted with $x$.

In [ ]:
A = np.array([[1, 2],
              [3, 4],
              [5, 6]])      # (3, 2)
x = np.array([1, 10])       # (2,)
print(A @ x)                # → [21 43 65]    row 0: 1·1 + 2·10 = 21, and so on
print(np.linalg.norm(np.array([3, 4])))   # → 5.0    length of a vector: √(3² + 4²)

The **norm** (length) of a vector is $\lVert v \rVert = \sqrt{\sum_i v_i^2}$, so $\lVert (3, 4) \rVert = \sqrt{9 + 16} = 5$.

**Eigenvectors.** For a square matrix $A$, an eigenvector $v$ is a direction that $A$ only **stretches** and doesn't rotate. The stretch factor $\lambda$ is the **eigenvalue**:

$$
A v = \lambda v \qquad \text{find } \lambda \text{ by solving } \det(A - \lambda I) = 0
$$

**By hand, $A = \begin{pmatrix} 2 & 1 \\ 1 & 2 \end{pmatrix}$:** $\det(A - \lambda I) = (2 - \lambda)^2 - 1 = 0 \Rightarrow 2 - \lambda = \pm 1 \Rightarrow \lambda = 1$ or $\lambda = 3$. For $\lambda = 3$, $(A - 3I)v = 0$ gives $-v_1 + v_2 = 0$, so $v \propto (1, 1)$. For $\lambda = 1$, $v \propto (1, -1)$.

In [ ]:
A = np.array([[2.0, 1.0],
              [1.0, 2.0]])
vals, vecs = np.linalg.eigh(A)     # eigh: for symmetric matrices, eigenvalues come out sorted
print(vals)                        # → [1. 3.]
v = vecs[:, 1]                     # eigenvector for λ = 3 (a COLUMN of vecs)
print(A @ v, 3 * v)                # A v equals 3 v: both ≈ [2.12 2.12]

**Why you care:** PCA (Week 4) is the eigenvectors of the data's **covariance matrix** $\Sigma$. The eigenvector with the biggest eigenvalue is the direction the data varies the most.

**SVD**, for any matrix: $A = U \Sigma V^\top$. For now just know that it exists, that it always works (even when $A$ isn't square), and that it's how PCA is computed in practice: `np.linalg.svd`.

**Drill (on paper).** Find the eigenvalues of $\begin{pmatrix} 4 & 0 \\ 0 & 1 \end{pmatrix}$ and of $\begin{pmatrix} 3 & 1 \\ 1 & 3 \end{pmatrix}$. Then compute $\begin{pmatrix} 1 & 2 \\ 3 & 4 \end{pmatrix} \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$.

<details><summary>Worked answer</summary>

First matrix: $\lambda = \mathbf{4}$ and $\mathbf{1}$ (a diagonal matrix's eigenvalues are its diagonal). Second: $(3 - \lambda)^2 = 1 \Rightarrow \lambda = \mathbf{4}, \mathbf{2}$. The product: row 0 is $(1 \cdot 0 + 2 \cdot 1,\ 1 \cdot 1 + 2 \cdot 0) = (2, 1)$, row 1 is $(3 \cdot 0 + 4 \cdot 1,\ 3 \cdot 1 + 4 \cdot 0) = (4, 3)$, so $\begin{pmatrix} 2 & 1 \\ 4 & 3 \end{pmatrix}$. Multiplying by this matrix swaps the columns.

</details>

## 7. Gradients and gradient descent

Training a model means **finding the weights that make the loss smallest**. The gradient $\nabla L$ points uphill, so we take small steps **downhill**, where $\eta$ (eta) is the **learning rate**:

$$
w \leftarrow w - \eta\, \nabla L(w)
$$

**One dimension, by hand.** $L(w) = (w - 3)^2$, so $L'(w) = 2(w - 3)$. Start at $w = 0$ with $\eta = 0.1$:

| Step | gradient $2(w - 3)$ | new $w = w - 0.1 \times$ gradient |
|---|---|---|
| $0 \to 1$ | $2(0 - 3) = -6$ | $0 + 0.6 = \mathbf{0.6}$ |
| $1 \to 2$ | $2(0.6 - 3) = -4.8$ | $0.6 + 0.48 = \mathbf{1.08}$ |
| $2 \to 3$ | $2(1.08 - 3) = -3.84$ | $1.08 + 0.384 = \mathbf{1.464}$ |

In [ ]:
w = 0.0
for step in range(50):
    grad = 2 * (w - 3)
    w = w - 0.1 * grad
print(round(w, 4))       # → 3.0    it reaches the minimum

**The least-squares gradient, the one result to remember.** For a linear model with predictions $Xw$, where $X \in \mathbb{R}^{n \times d}$ holds one sample $x_i^\top$ per row:

$$
L(w) = \lVert Xw - y \rVert^2 = \sum_{i=1}^{n} (x_i^\top w - y_i)^2 \qquad \nabla L(w) = 2 X^\top (Xw - y)
$$

**Where it comes from:** for one sample, $\frac{\partial}{\partial w}(x_i^\top w - y_i)^2 = 2(x_i^\top w - y_i)\, x_i$ by the chain rule (the inside derivative of $x_i^\top w$ with respect to $w$ is $x_i$). Summing over the samples stacks those $x_i$'s as the columns of $X^\top$: $\sum_i 2(x_i^\top w - y_i)\, x_i = 2X^\top(Xw - y)$.

Setting the gradient to zero gives the **normal equation**, the closed-form solution of linear regression (Week 2):

$$
X^\top X\, w = X^\top y \quad\Longrightarrow\quad \hat w = (X^\top X)^{-1} X^\top y
$$

Check both claims numerically:

In [ ]:
X = rng.normal(size=(20, 3))
y = rng.normal(size=20)
w = rng.normal(size=3)
loss = lambda w: np.sum((X @ w - y) ** 2)

analytic = 2 * X.T @ (X @ w - y)
eps = 1e-6
numeric = np.array([(loss(w + eps * e) - loss(w - eps * e)) / (2 * eps) for e in np.eye(3)])
print(np.allclose(analytic, numeric))                       # → True   the formula is right

w_best = np.linalg.solve(X.T @ X, X.T @ y)                  # solve the normal equation
print(np.allclose(2 * X.T @ (X @ w_best - y), 0))           # → True   gradient is 0 there

**Drill (on paper).** $f(w) = w^2$, starting at $w = 1$, learning rate $\eta = 0.25$. Do two gradient-descent steps by hand.

<details><summary>Worked answer</summary>

$f'(w) = 2w$. Step 1: $w = 1 - 0.25 \times 2 = \mathbf{0.5}$. Step 2: $w = 0.5 - 0.25 \times 1 = \mathbf{0.25}$. Each step halves $w$.

</details>

## 8. Overfitting, bias–variance, and train/validation/test

A model that is too **simple** misses the pattern (**underfitting**, high bias). A model that is too **flexible** memorizes the noise (**overfitting**, high variance). See it with polynomials of different degree fitted to noisy data:

In [ ]:
x_train = np.linspace(0, 1, 15);  y_train = np.sin(2 * np.pi * x_train) + rng.normal(0, 0.2, 15)
x_test  = np.linspace(0, 1, 200); y_test  = np.sin(2 * np.pi * x_test)  + rng.normal(0, 0.2, 200)
for degree in [1, 4, 12]:
    coefs = np.polyfit(x_train, y_train, degree)
    mse = lambda x, y: np.mean((np.polyval(coefs, x) - y) ** 2)
    print(f"degree {degree:2d}: train MSE {mse(x_train, y_train):.3f}   test MSE {mse(x_test, y_test):.3f}")
# → degree  1: train MSE 0.258   test MSE 0.253    both high: underfit
#   degree  4: train MSE 0.027   test MSE 0.053    both low: good
#   degree 12: train MSE 0.003   test MSE 0.117    train tiny, test 2× worse: overfit

**The bias–variance decomposition.** For a prediction $\hat f(x)$ of $y = f(x) + \varepsilon$ with noise variance $\sigma^2$:

$$
\mathbb{E}\big[(y - \hat f(x))^2\big] = \underbrace{\big(\mathbb{E}[\hat f(x)] - f(x)\big)^2}_{\text{bias}^2} + \underbrace{\operatorname{Var}\big(\hat f(x)\big)}_{\text{variance}} + \underbrace{\sigma^2}_{\text{noise}}
$$

More flexibility lowers the bias but raises the variance, and the best model is in the middle.

| Split | Used for | Rule |
|---|---|---|
| train | fitting the weights | the model sees it |
| validation | choosing settings: degree, $\lambda$, learning rate $\eta$ | used many times, while developing |
| test | the final, honest score | touch it **once**, at the very end |

**Data leakage** means information from the validation or test data sneaking into training (e.g. standardizing with the mean of *all* the data before splitting). Your KAMP copied-days finding was a real example of this.

## 9. Entropy, cross-entropy, KL divergence

**Entropy** measures uncertainty. A fair coin has $H = 1$ bit (using $\log_2$). A coin that always lands heads has $H = 0$, since there's no uncertainty. Decision trees (Week 3) split data to reduce entropy.

$$
H(p) = -\sum_i p_i \log p_i
$$

**Cross-entropy** compares the true distribution $p$ with the model's prediction $q$. In classification, $p$ is **one-hot** (1 for the true class, 0 for the rest), so it simplifies to just $-\log q_{\text{true}}$. That's the classification loss in every neural network:

$$
H(p, q) = -\sum_i p_i \log q_i \;\overset{\text{one-hot } p}{=}\; -\log q_{\text{true}}
$$

In [ ]:
def cross_entropy(true_class, q):
    return -np.log(q[true_class])

print(round(cross_entropy(2, np.array([0.1, 0.2, 0.7])), 3))   # → 0.357   confident and right: small loss
print(round(cross_entropy(2, np.array([0.6, 0.3, 0.1])), 3))   # → 2.303   confident and wrong: big loss

**KL divergence** measures how far $q$ is from $p$. It's always $\ge 0$, and it's $0$ only when $q = p$. It appears in VAEs (Week 7):

$$
D_{\mathrm{KL}}(p \,\Vert\, q) = \sum_i p_i \log \frac{p_i}{q_i} = H(p, q) - H(p) \;\ge\; 0
$$

In [ ]:
p = np.array([0.5, 0.5]); q = np.array([0.9, 0.1])
H = lambda p: -np.sum(p * np.log2(p))
print(H(p), round(H(q), 3))                       # → 1.0 0.469
KL = np.sum(p * np.log(p / q))
print(round(KL, 3))                               # → 0.511

**Drill (on paper).** True class 0, prediction $q = (0.5, 0.25, 0.25)$. What is the cross-entropy loss (natural log)? What is the entropy of $q$ in bits?

<details><summary>Worked answer</summary>

$-\ln 0.5 = \mathbf{0.693}$. $H(q) = -(0.5 \log_2 0.5 + 2 \times 0.25 \log_2 0.25) = 0.5 \times 1 + 0.5 \times 2 = \mathbf{1.5}$ bits.

</details>

---
The Sunday quiz (**`05_sunday_quiz.ipynb`**) has 5 math questions like these drills. Log any drill you got wrong.